In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import warnings

/mnt/windows/wheelchair-dev/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = 'huawei-noah/TinyBERT_General_4L_312D'
THRESHOLD = 0.5
Y_MIN, Y_MAX = 0.0, 7.58
SIZE = 32
MAX_LEN = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:

# Labelling Regression score
def score_to_category(score):
    if score <= 0.20:   return "Safe"
    elif score <= 0.45: return "Ambiguous"
    elif score <= 0.75: return "Implicit/Covert Hate"
    else:               return "Explicit Toxicity"

In [4]:
df = pd.read_csv('../data/processed/train_cleaned.csv')
print(f"Loaded {len(df)} rows | target range: {df['target'].min():.4f}–{df['target'].max():.4f}")


df['category'] = df['target'].apply(score_to_category)
print(df['category'].value_counts())
df.head(5)

Loaded 31225 rows | target range: 0.0000–1.0000
category
Safe                    15618
Ambiguous                8899
Implicit/Covert Hate     4481
Explicit Toxicity        2227
Name: count, dtype: int64


,text,cleaned_text,target,toxic,severe_toxic,obscene,threat,insult,identity_hate,category
0,"""==dont leave==\nWikipedia needs people like y...",dont leave Wikipedia needs people like you Rem...,0.00,0,0,0,0,0,0,Safe
1,First problem ParacusForward like most wiki ed...,First problem ParacusForward like most wiki ed...,0.25,1,0,0,0,0,0,Ambiguous
2,"Grow up, cyber yuppie.",Grow up cyber yuppie,0.25,1,0,0,0,0,0,Ambiguous
3,"""\nThis is ridiculous. I can't respond BECAUS...",This is ridiculous I cant respond BECAUSE YOU ...,0.00,0,0,0,0,0,0,Safe
4,Here is proof of notability \nCalton has aimed...,Here is proof of notability Calton has aimed h...,0.25,1,0,0,0,0,0,Ambiguous


In [5]:
class ToxicityDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts    = texts.reset_index(drop=True)
        self.targets  = targets.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'target':         torch.tensor(self.targets[idx], dtype=torch.float)
        }

In [6]:
class TinyBERTToxicityModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        
        self.encoder = AutoModel.from_pretrained(
            model_name,
            attn_implementation="eager"
        )
        hidden = self.encoder.config.hidden_size

        self.regressor = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.identity_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           output_attentions=True)
        cls_emb    = out.last_hidden_state[:, 0, :]
        toxicity   = self.regressor(cls_emb).squeeze(-1)
        identity   = self.identity_classifier(cls_emb).squeeze(-1)
        attentions = out.attentions
        return toxicity, identity, attentions


def ear_penalty(attentions, lambda_ear=0.001):
    penalty = 0.0
    for layer_attn in attentions:
        attn = layer_attn.clamp(min=1e-9)
        entropy = -(attn * attn.log()).sum(-1)  
        penalty += entropy.mean()              
    penalty = penalty / len(attentions)         
    return -lambda_ear * penalty               

In [7]:
ACCUMULATION_STEPS = 4
scaler = torch.cuda.amp.GradScaler()

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        targets   = batch['target'].to(device)  # keep float32

        with torch.amp.autocast('cuda'):
            toxicity, identity, attentions = model(input_ids, attn_mask)
            loss_mse = weighted_mse(toxicity.float(), targets.float())
            loss_ear = ear_penalty(attentions)
            loss     = (loss_mse + loss_ear) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS

    return total_loss / len(loader)


def eval_epoch(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            targets   = batch['target'].to(device)

            with torch.amp.autocast('cuda'):
                toxicity, _, _ = model(input_ids, attn_mask)

            all_preds.extend(toxicity.cpu().float().numpy())
            all_targets.extend(targets.cpu().float().numpy())

    preds   = np.array(all_preds)
    targets = np.array(all_targets)
    mse     = mean_squared_error(targets, preds)

    bin_preds   = (preds   >= THRESHOLD).astype(int)
    bin_targets = (targets >= THRESHOLD).astype(int)
    precision = precision_score(bin_targets, bin_preds, zero_division=0)
    recall    = recall_score(bin_targets, bin_preds, zero_division=0)
    f1        = f1_score(bin_targets, bin_preds, zero_division=0)

    return mse, precision, recall, f1

/tmp/ipykernel_342844/3277931943.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [8]:
print(torch.cuda.get_device_name(0))
print(f"VRAM total:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free:   {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")

NVIDIA GeForce RTX 3050 Laptop GPU
VRAM total:  4.0 GB
VRAM free:   0.0 GB reserved


In [9]:
print(DEVICE)  # if this says 'cpu', that's your problem
print(torch.cuda.is_available())

cuda
True


In [10]:
import gc
from torch.utils.data import WeightedRandomSampler

# Clear GPU memory
if 'model' in dir():
    del model
torch.cuda.empty_cache()
gc.collect()


df_continuous  = df[(df['target'] > 0.001) & (df['target'] < 0.999)].copy()
df_safe_sample = df[df['target'] <= 0.001].sample(n=4000, random_state=42)
df_clean = pd.concat([df_continuous, df_safe_sample]).sample(frac=1, random_state=42)
df_clean['category'] = df_clean['target'].apply(score_to_category)

print("Clean dataset distribution:")
print(df_clean['category'].value_counts())
print(f"Total: {len(df_clean)} rows")

MODEL_NAME = 'distilbert-base-uncased'

train_df, val_df = train_test_split(df_clean, test_size=0.15,
                                    random_state=42, stratify=df_clean['category'])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = ToxicityDataset(train_df['cleaned_text'], train_df['target'], tokenizer, MAX_LEN)
val_ds   = ToxicityDataset(val_df['cleaned_text'],   val_df['target'],   tokenizer, MAX_LEN)


category_counts = train_df['category'].value_counts()
sample_weights  = train_df['category'].map(lambda c: 1.0 / category_counts[c]).values
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_ds),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=16,
                          num_workers=2, pin_memory=True)


def weighted_mse(preds, targets, threshold_implicit=0.45, threshold_explicit=0.75):
    weights = torch.ones_like(targets)
    weights = torch.where(targets >= threshold_implicit,
                          torch.full_like(targets, 5.0), weights)
    weights = torch.where(targets >= threshold_explicit,
                          torch.full_like(targets, 15.0), weights)
    return (weights * (preds - targets) ** 2).mean()


model     = TinyBERTToxicityModel(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler    = torch.cuda.amp.GradScaler()


best_val_mse     = float('inf')
patience         = 5
patience_counter = 0
best_epoch       = 0

EPOCHS = 50
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, DEVICE)
    mse, prec, rec, f1 = eval_epoch(model, val_loader, DEVICE)
    scheduler.step()
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val MSE: {mse:.4f} | Precision: {prec:.4f} | "
          f"Recall: {rec:.4f} | F1: {f1:.4f}")

    if mse < best_val_mse:
        best_val_mse     = mse
        best_epoch       = epoch
        patience_counter = 0
        torch.save(model.state_dict(), '../models/distilbert_toxicity_best.pt')
        print(f"  New best saved (val MSE: {mse:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}. Best was epoch {best_epoch}.")
            break

model.load_state_dict(torch.load('../models/distilbert_toxicity_best.pt',
                                  map_location=DEVICE, weights_only=True))
print("Best model loaded.")

Clean dataset distribution:
category
Ambiguous               8899
Safe                    4618
Implicit/Covert Hate    4481
Explicit Toxicity       1778
Name: count, dtype: int64
Total: 19776 rows


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6992.14it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_342844/15376633.py:57: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


Epoch 1/50 | Train Loss: 0.1864 | Val MSE: 0.0573 | Precision: 0.5971 | Recall: 0.8647 | F1: 0.7064
  New best saved (val MSE: 0.0573)
Epoch 2/50 | Train Loss: 0.1037 | Val MSE: 0.0344 | Precision: 0.7026 | Recall: 0.7497 | F1: 0.7254
  New best saved (val MSE: 0.0344)
Epoch 3/50 | Train Loss: 0.0730 | Val MSE: 0.0476 | Precision: 0.6175 | Recall: 0.8701 | F1: 0.7224
  No improvement (1/5)
Epoch 4/50 | Train Loss: 0.0572 | Val MSE: 0.0385 | Precision: 0.6562 | Recall: 0.8232 | F1: 0.7303
  No improvement (2/5)
Epoch 5/50 | Train Loss: 0.0458 | Val MSE: 0.0386 | Precision: 0.6618 | Recall: 0.8147 | F1: 0.7303
  No improvement (3/5)
Epoch 6/50 | Train Loss: 0.0441 | Val MSE: 0.0386 | Precision: 0.6618 | Recall: 0.8147 | F1: 0.7303
  No improvement (4/5)
Epoch 7/50 | Train Loss: 0.0432 | Val MSE: 0.0378 | Precision: 0.6705 | Recall: 0.8083 | F1: 0.7330
  No improvement (5/5)
Early stopping at epoch 7. Best was epoch 2.
Best model loaded.


In [11]:
def predict(text, model, tokenizer, device):
    model.eval()
    enc = tokenizer(text, return_tensors='pt', max_length=MAX_LEN,
                    truncation=True, padding='max_length')
    with torch.no_grad():
        score, _, _ = model(enc['input_ids'].to(device),
                            enc['attention_mask'].to(device))
    score = score.item()
    return {
        "raw_score":  score,
        "y_original": score * (Y_MAX - Y_MIN) + Y_MIN,
        "category":   score_to_category(score),
        "flag":       score >= THRESHOLD
    }



In [12]:
# Sanity Checks
tests = [
    # Explicit Toxicity (>0.75)
    ("eat shit fucko suck my asshole then die horribly i hope you get cancer",
     "expect >0.75 Explicit"),

    # Implicit/Covert Hate (0.46–0.75)
    ("I absolutely hate you, you disgusting piece of trash.",
     "expect 0.46–0.75 Implicit"),

    # Ambiguous (0.21–0.45)
    ("grandma terri die in trash if cking hate you",
     "expect 0.21–0.45 Ambiguous"),

    # Safe (<0.20) identity term, neutral context
    ("I had a really wonderful day at the park today.",
     "expect <0.20 Safe"),

    # Safe (<0.20)
    ("My new neighbor is a Muslim man.",
     "expect <0.20 Safe"),

    # Safe (<0.20) negation
    ("I am not a hateful person, and I do not despise you.",
     "expect <0.20 Safe"),
]

for text, expectation in tests:
    r = predict(text, model, tokenizer, DEVICE)
    status = "✓" if (
        ("0.75" in expectation and r['raw_score'] > 0.75) or
        ("0.46" in expectation and 0.46 <= r['raw_score'] <= 0.75) or
        ("0.21" in expectation and 0.21 <= r['raw_score'] <= 0.45) or
        ("Safe" in expectation and r['raw_score'] < 0.20)
    ) else "✗"
    print(f"{status} [{r['category']:25s}] score={r['raw_score']:.3f} | {expectation}")
    print(f"  → {text[:70]}")

✓ [Explicit Toxicity        ] score=0.931 | expect >0.75 Explicit
  → eat shit fucko suck my asshole then die horribly i hope you get cancer
✓ [Implicit/Covert Hate     ] score=0.572 | expect 0.46–0.75 Implicit
  → I absolutely hate you, you disgusting piece of trash.
✗ [Implicit/Covert Hate     ] score=0.589 | expect 0.21–0.45 Ambiguous
  → grandma terri die in trash if cking hate you
✗ [Ambiguous                ] score=0.214 | expect <0.20 Safe
  → I had a really wonderful day at the park today.
✗ [Ambiguous                ] score=0.359 | expect <0.20 Safe
  → My new neighbor is a Muslim man.
✗ [Ambiguous                ] score=0.411 | expect <0.20 Safe
  → I am not a hateful person, and I do not despise you.


In [21]:
examples = [
    'Honestly you should go fuck yourself',
    'You should just die of cancer',
    'Fucking whore',
    'You are the cutest being on earth',
    'Elituqinn is the best Osu Player in Indonesia',
    "Your Character is so strong",
    "Wow, Skott! I am genuinely impressed. I didn't think it was humanly possible to fit that much ego into one single suit, but look at you—breaking records every day!",
    'I love you in whole wide world'
]

for text in examples:
    result = predict(text, model, tokenizer, DEVICE)
    print(f"Text : {text} : Result :  {result}")


Text : Honestly you should go fuck yourself : Result :  {'raw_score': 0.7913423776626587, 'y_original': 5.998375222682953, 'category': 'Explicit Toxicity', 'flag': True}
Text : You should just die of cancer : Result :  {'raw_score': 0.8126732110977173, 'y_original': 6.160062940120697, 'category': 'Explicit Toxicity', 'flag': True}
Text : Fucking whore : Result :  {'raw_score': 0.9139068126678467, 'y_original': 6.927413640022278, 'category': 'Explicit Toxicity', 'flag': True}
Text : You are the cutest being on earth : Result :  {'raw_score': 0.4261929392814636, 'y_original': 3.2305424797534945, 'category': 'Ambiguous', 'flag': False}
Text : Elituqinn is the best Osu Player in Indonesia : Result :  {'raw_score': 0.2469864934682846, 'y_original': 1.8721576204895973, 'category': 'Ambiguous', 'flag': False}
Text : Your Character is so strong : Result :  {'raw_score': 0.23404008150100708, 'y_original': 1.7740238177776337, 'category': 'Ambiguous', 'flag': False}
Text : Wow, Skott! I am genuin